# Equilibrium teacher data and the six students of Li$_3$OCl

1. equilibrium teacher dynamics at 1000, 900, 800 and 700 K (32 replicas × 8 ps each, a frame every 0.25 ps) with teacher energies and forces: the training pool and the basin frames
2. training of the six students of Sec. VI C of the paper (`mace_run_train`)
3. held-out errors and speed of every student

Input: `step5_bundle.zip`, as in step 5. Runtime: GPU. With `USE_DRIVE = True` finished parts are skipped when all cells are run again. Output: the student models `li3ocl_S1.model` to `li3ocl_S6.model`, the equilibrium frames `eq_*K.npz`, and `pool.xyz` and `test.xyz`.

In [ ]:
!pip -q install mace-torch ase
import torch, time, json, os, glob, subprocess, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
USE_DRIVE = True; OUT = '.'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/li3ocl_step6'; os.makedirs(OUT, exist_ok=True)
    except Exception as err: print('no Drive, working locally:', err)
if not os.path.exists('batched_md.py'):
    from google.colab import files
    up = files.upload()            # choose step5_bundle.zip
    os.system('unzip -o -q step5_bundle.zip')
print(OUT, sorted(os.listdir('.')))

In [ ]:
# (1) equilibrium teacher frames with labels
from ase import Atoms
from ase.io import write, read
from mace.calculators import MACECalculator
from batched_md import BatchedMD
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; A = 3.926; B = 32
TEMPS = [1000.0, 900.0, 800.0, 700.0]; EQUIL, PROD, EVERY = 1000, 4000, 125          # steps of 2 fs: 2 ps, 8 ps, a frame every 0.25 ps
unit = Atoms('ClOLi3', scaled_positions=[(0,0,0), (.5,.5,.5), (.5,.5,0), (.5,0,.5), (0,.5,.5)], cell=[A]*3, pbc=True)
s = unit.repeat((3,3,3)); li_all = [i for i, z in enumerate(s.get_chemical_symbols()) if z == 'Li']; del s[li_all[0]]
calc = MACECalculator(model_paths='li3ocl_teacher.model', device=DEV, default_dtype='float32'); model = calc.models[0]
t_start = time.time()
for T in TEMPS:
    fn = f'{OUT}/eq_{int(T)}K.npz'
    if os.path.exists(fn): print(fn, 'exists'); continue
    rng = np.random.default_rng(int(T) + 1); x0 = np.stack([s.positions + rng.normal(0, 0.05, s.positions.shape) for _ in range(B)])
    eng = BatchedMD(model, s.numbers, s.cell.lengths(), x0, T, device=DEV, seed=int(T) + 1); eng.run(EQUIL); X, E, F = [], [], []
    def grab(e): X.append(e.x.cpu().numpy().astype(np.float32)); E.append(e.e.cpu().numpy().astype(np.float64)); F.append(e.f.cpu().numpy().astype(np.float32))
    eng.step_count = 0; eng.run(PROD, callback=grab, callback_every=EVERY)
    X, E, F = (np.concatenate(a) for a in (X, E, F)); np.savez_compressed(fn, X=X, E=E, F=F, T=T, numbers=s.numbers, cell=s.cell.lengths())
    print(f'{T:.0f} K: {len(E)} frames, <T> = {eng.temperature().mean():.0f} K, E sd {E.std():.3f} eV, {(time.time() - t_start) / 60:.0f} min', flush=True)

In [ ]:
# (2) training and test files (fixed permutation, seed 0), then the six students
def frames(idx_by_T):
    out = []
    for T, idx in idx_by_T:
        d = np.load(f'{OUT}/eq_{int(T)}K.npz')
        for i in idx:
            a = s.copy(); a.positions = d['X'][i]; a.wrap(); a.info['REF_energy'] = float(d['E'][i]); a.arrays['REF_forces'] = d['F'][i].astype(float); a.info['T'] = T; out.append(a)
    return out
if not os.path.exists(f'{OUT}/pool.xyz'):
    rng = np.random.default_rng(0); test, pool = [], []
    for T in TEMPS:
        n = len(np.load(f'{OUT}/eq_{int(T)}K.npz')['E']); p = rng.permutation(n); test.append((T, p[:100])); pool.append((T, p[100:]))
    pl = frames(pool); order = rng.permutation(len(pl)); write(f'{OUT}/pool.xyz', [pl[i] for i in order]); write(f'{OUT}/test.xyz', frames(test))
pool = read(f'{OUT}/pool.xyz', ':'); print(len(pool), 'pool frames;', len(read(f'{OUT}/test.xyz', ':')), 'test frames')
STUDENTS = {'S1': ('32x0e', 5.0, 1000, 1), 'S2': ('32x0e', 5.0, 1000, 2), 'S3': ('16x0e', 5.0, 1000, 1), 'S4': ('32x0e', 4.0, 1000, 1), 'S5': ('32x0e', 5.0, 200, 1), 'S6': ('16x0e', 4.0, 200, 1)}
EPOCHS = 60
for name, (irreps, rmax, N, seed) in STUDENTS.items():
    if os.path.exists(f'{OUT}/li3ocl_{name}.model'): print(name, 'exists'); continue
    write(f'train_{name}.xyz', pool[:N]); t0 = time.time()
    cmd = (f"mace_run_train --name=li3ocl_{name} --train_file=train_{name}.xyz --valid_fraction=0.08 --test_file={OUT}/test.xyz --energy_key=REF_energy --forces_key=REF_forces "
           f"--E0s=average --model=MACE --hidden_irreps={irreps} --r_max={rmax} --num_interactions=2 --correlation=3 --batch_size=8 --valid_batch_size=8 --max_num_epochs={EPOCHS} "
           f"--lr=0.01 --ema --ema_decay=0.99 --amsgrad --forces_weight=100 --energy_weight=1 --device={DEV} --default_dtype=float32 --seed={seed} --save_cpu")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print(name, 'exit', r.returncode, f'{(time.time() - t0) / 60:.0f} min'); print('\n'.join((r.stdout + r.stderr).splitlines()[-6:]), flush=True)
    if os.path.exists(f'li3ocl_{name}.model') and OUT != '.': os.system(f'cp li3ocl_{name}.model {OUT}/')

In [ ]:
# (3) held-out errors (force RMSE, energy RMSE after removing the mean offset) and MD speed
test = read(f'{OUT}/test.xyz', ':'); Eref = np.array([a.info['REF_energy'] for a in test]); Fref = np.array([a.arrays['REF_forces'] for a in test]); REPORT = {}
for name in STUDENTS:
    fn = f'{OUT}/li3ocl_{name}.model'
    if not os.path.exists(fn): REPORT[name] = 'training failed'; continue
    c = MACECalculator(model_paths=fn, device=DEV, default_dtype='float32'); E, F = [], []
    for a in test: b = a.copy(); b.calc = c; E.append(b.get_potential_energy()); F.append(b.get_forces())
    dE = np.array(E) - Eref; dF = np.array(F) - Fref
    x0 = np.stack([s.positions + np.random.default_rng(b).normal(0, 0.05, s.positions.shape) for b in range(B)]); eng = BatchedMD(c.models[0], s.numbers, s.cell.lengths(), x0, 1000.0, device=DEV, seed=1); eng.run(20)
    if DEV == 'cuda': torch.cuda.synchronize()
    t0 = time.time(); eng.run(50)
    if DEV == 'cuda': torch.cuda.synchronize()
    ms = (time.time() - t0) / 50 * 1e3
    REPORT[name] = dict(irreps=STUDENTS[name][0], r_max=STUDENTS[name][1], N=STUDENTS[name][2], seed=STUDENTS[name][3], force_rmse_meVA=round(float(np.sqrt((dF ** 2).mean()) * 1e3), 1),
                        energy_rmse_meV_cell=round(float((dE - dE.mean()).std() * 1e3), 1), energy_offset_meV_cell=round(float(dE.mean() * 1e3), 1), ms_per_replica_step=round(ms / B, 2), T_after_140fs=round(float(eng.temperature().mean())))
    print(name, REPORT[name], flush=True)
json.dump(REPORT, open(f'{OUT}/students_report.json', 'w'), indent=1)
print('=========== SUMMARY ==========='); print(json.dumps(dict(gpu=torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu', students=REPORT), indent=1)); print('==============================')
os.system(f'cd {OUT} && zip -q li3ocl_step6_results.zip li3ocl_S*.model eq_*K.npz test.xyz students_report.json && cp li3ocl_step6_results.zip /content/ 2>/dev/null')
from google.colab import files; files.download('/content/li3ocl_step6_results.zip' if os.path.exists('/content/li3ocl_step6_results.zip') else f'{OUT}/li3ocl_step6_results.zip')